<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_3/notebooks/W10L1_Functional_imaging_analysis__STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# W10 L1 — Functional Imaging: Does 4-AP Make Neurons Fire More?

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**  ·  *In-class activity*

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

---

## Background

In W5 you blocked **Na⁺ channels** with lidocaine and watched a model neuron fire **less**. Today you
test a drug expected to do the opposite.

**4-aminopyridine (4-AP)** blocks voltage-gated **K⁺ channels**. Recall from W5 that the K⁺ current is
what ends each action potential and pulls the membrane back down. Take some of it away and neurons
repolarise more slowly, sit closer to threshold, and fire **more readily**. 4-AP is so reliably
excitatory that labs use it to provoke seizure-like activity in brain slices.

So the prediction is clear: **4-AP should make neurons fire more often.** The question is how you
would *see* that in living tissue, where you cannot put an electrode into a hundred cells at once.

The answer is **calcium imaging**. Neurons are loaded with a dye that gets brighter when calcium
floods in, and calcium floods in through voltage-gated Ca²⁺ channels every time a neuron fires. So
brightness over time is a readable trace of activity — a cell that lights up is a cell that just fired.

## Your experiment

Two brain slices were imaged, **100 neurons** each, for **6 minutes at 10 frames per second**. Three
minutes in, a solution was washed onto each slice:

| Group | What was washed on at 180 s | What to expect |
|---|---|---|
| **CONTROL** | **aCSF** — artificial cerebrospinal fluid, the vehicle, no drug | activity should barely change |
| **DRUG** | **4-AP** | activity should **rise**, if the prediction is right |

Each recording has a **baseline period (0–180 s)** before wash-on and an **application period
(180–360 s)** after it. Imaging software (**Suite2p**, **CaImAn**, **ImageJ**) found each neuron and
measured its brightness in every frame, so the movie became a CSV with **one row per frame and one
column per cell** (`C1`–`C100`).

The dataset is realistic but **simulated**, so everyone works with identical numbers.

## What you'll be able to do by the end

1. Explain what a **frame**, the **frame rate**, and the **baseline** vs **application** periods are
2. Compute **ΔF/F₀**, and say why raw brightness can't be compared between cells
3. Read a **heatmap** and a **trace plot**, and say what each one shows that the other hides
4. Detect **calcium events** with a threshold and turn them into an **event rate**
5. Judge how much your conclusion depended on the **threshold** you chose

## How this notebook works

**Run every cell in order, from the top.** Click a cell and press **Shift + Enter**.

| | What to do |
|---|---|
| **Ordinary cells** | Just run them. You don't need to understand every line of code. |
| **⚙️ Try it** | At most one per step: a value in the code that **you change and re-run**. |
| **💬 Discussion** | At the very end. Answer using the numbers you found along the way. |

If something breaks, first check that you ran **every cell above it**.

**To submit:** answer the Discussion, click **Runtime → Run all**, then **File → Print → Save as PDF**
and upload the PDF to Canvas.

---

## Step 1 — Load the data and look at raw brightness

This cell loads the dataset straight from the course GitHub page — nothing to download. `numpy` does
the maths, `pandas` holds the table, and `matplotlib` draws the figures.

`FPS` (frames per second) is how the recording was acquired: at 10 fps there is one frame every 0.1 s,
which is what lets you convert frame counts into seconds, and seconds into a rate per minute in Step 4.

In [ ]:
import os
import numpy as np                # for working with arrays of numbers
import pandas as pd               # for the data table
import matplotlib.pyplot as plt   # for every figure in this notebook

DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_3/data/"

FPS = 10          # frames per second: one frame every 0.1 s
N_CELLS = 100     # cells per recording (the columns C1..C100)

# Load the table, falling back to a hand-uploaded copy if the download fails
FILENAME = "functional_imaging_dummy.csv"
try:
    df = pd.read_csv(DATA + FILENAME)
except Exception:
    if not os.path.exists(FILENAME):
        raise SystemExit(
            "\n*** Could not download " + FILENAME + ". ***\n"
            "Click the folder icon in the left sidebar, upload " + FILENAME + ",\n"
            "then run this cell again.")
    df = pd.read_csv(FILENAME)

# The cell columns are the ones named C1, C2, ... C100. We reuse this list constantly.
cell_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]

print("Combined table (rows, columns):", df.shape)
print("Rows per recording:            ", df.groupby("Group").size().to_dict())
print("Cells per recording:           ", len(cell_cols))
print("Recording length:              ", round(df["Time"].max() + 1 / FPS), "s")

df.head()

Each number in that table is how much light one cell gave off in one frame. It is **not** a measure of
activity on its own: one cell can be far brighter than another simply because it took up more dye or
sat closer to the focal plane. The cell below plots one cell's raw brightness and prints the
**resting** brightness of a few cells, so you can see how much they differ before anything happens.

> **⚙️ Try it — inspect different cells.** Run with `CELL_TO_INSPECT = "C1"`, then try `"C4"` and
> `"C5"`. Write down each one's resting brightness. Set it back to `"C1"`.

In [ ]:
CELL_TO_INSPECT = "C1"    # ⚙️ Try it — any cell from "C1" to "C100"

fig, ax = plt.subplots(figsize=(11, 3.5))
for name, colour in [("CONTROL", "gray"), ("DRUG", "tomato")]:
    sub = df[df["Group"] == name]
    ax.plot(sub["Time"], sub[CELL_TO_INSPECT], color=colour, lw=0.6,
            label="%s recording, cell %s" % (name, CELL_TO_INSPECT))
ax.axvline(180, color="black", ls="--", lw=1.2)
ax.text(181, ax.get_ylim()[1], " wash-on", va="top", fontsize=9)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Raw fluorescence (a.u.)")
ax.set_title("Raw brightness of cell %s" % CELL_TO_INSPECT)
ax.legend(fontsize=8, loc="upper left")
plt.show()

# Resting brightness = the average raw value during the baseline (first 180 s)
resting = df[(df["Group"] == "DRUG") & (df["Time"] < 180)][cell_cols].mean()
print("Resting raw brightness in the DRUG recording:")
print("   %-4s %.0f   <- the cell you are inspecting" % (CELL_TO_INSPECT, resting[CELL_TO_INSPECT]))
print("   dimmest cell:   %s  %.0f" % (resting.idxmin(), resting.min()))
print("   brightest cell: %s  %.0f" % (resting.idxmax(), resting.max()))

## Step 2 — Convert brightness into ΔF/F₀

Because raw brightness can't be compared between cells, we measure how much each cell changes from
**its own** resting brightness. That relative change is **ΔF/F₀** ("delta-F over F-zero"):

- **F** — the cell's brightness at one frame
- **F₀** — the cell's **resting** brightness: its average over the **baseline period** (first 180 s)
- **ΔF** — the change from rest, F − F₀

$$\frac{\Delta F}{F_0} = \frac{F - F_0}{F_0}\times 100\%$$

A cell resting at F₀ = 1000 that brightens to F = 1400 has ΔF/F₀ = 400 ÷ 1000 × 100 = **+40%**. A quiet
cell has F = F₀, so ΔF/F₀ = **0%**. A cell dimmer than its own rest has a **negative** ΔF/F₀. Because each
cell is measured against **its own** F₀, a dim cell and a bright cell finally sit on the same scale.

The baseline ΔF/F₀ must come out at **≈ 0%** — that is a check that the conversion worked, not a
result. The **application** numbers are the ones to read.

> **⚙️ Try it — move the F₀ window.** Run with `F0_WINDOW_SEC = 180`, then set it to `360` (the whole
> recording, drug period included) and re-run. Watch the **DRUG baseline** number. Set it back to
> **180** and re-run before moving on.

In [ ]:
F0_WINDOW_SEC = 180     # ⚙️ Try it — the window (s) that defines F0. Try 360, then set back to 180.


def compute_dff(group_name):
    """Return a Time + ΔF/F₀ table for one recording (CONTROL or DRUG)."""
    sub      = df[df["Group"] == group_name].reset_index(drop=True)
    baseline = sub[sub["Time"] < F0_WINDOW_SEC]      # the frames that define rest
    F0       = baseline[cell_cols].mean()            # each cell's own resting brightness
    dff      = (sub[cell_cols] - F0) / F0 * 100      # (F - F0) / F0, as a percentage
    return pd.concat([sub[["Time"]], dff], axis=1)   # keep Time alongside the signal


dff_con  = compute_dff("CONTROL")
dff_drug = compute_dff("DRUG")

print("F0_WINDOW_SEC = %g s\n" % F0_WINDOW_SEC)
print("Average ΔF/F₀ in each half of each recording:")
for name, d in [("CONTROL", dff_con), ("DRUG", dff_drug)]:
    base = d.loc[d["Time"] <  180, cell_cols].values.mean()
    app  = d.loc[d["Time"] >= 180, cell_cols].values.mean()
    print("   %-8s baseline = %5.2f %%   application = %5.2f %%" % (name, base, app))

## Step 3 — Look at the whole population, then at single cells

Two views of the same data, because each one hides what the other shows.

A **heatmap** puts all 100 cells × 3600 frames on screen at once: each **row is one cell**, the
**x-axis is time**, and **colour is ΔF/F₀**. It shows *when* activity happened and *how many cells*
were involved. Before the dashed line the two panels are two samples of the same thing, so they
should look alike.

A **trace plot** draws lines instead, which is how you see the **shape** of a calcium event — a sharp
rise as calcium enters, then a slower decay as the cell pumps it back out. Twelve cells are stacked
with a vertical offset; the offset is arbitrary, so the y-axis has no scale.

`COLOUR_MAX` sets which ΔF/F₀ value counts as full brightness in the heatmap. Both panels always share
it, but the number you choose changes how dramatic the figure looks.

> **⚙️ Try it — change the colour scale.** Run at `COLOUR_MAX = 60`, then `20`, then `150`. Did any
> **number** in the data change? Set it back to **60**.

In [ ]:
COLOUR_MAX = 60      # ⚙️ Try it — the ΔF/F₀ (%) shown as full brightness. Try 20, then 150.

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (name, d) in zip(axes, [("CONTROL", dff_con), ("DRUG", dff_drug)]):
    # .T puts cells on rows and time on columns; vmin/vmax keep both panels on one colour scale
    im = ax.imshow(d[cell_cols].values.T, aspect="auto", cmap="viridis",
                   vmin=-20, vmax=COLOUR_MAX, extent=[0, d["Time"].max(), N_CELLS, 1])
    ax.axvline(180, color="white", linestyle="--", linewidth=1.5)   # wash-on
    ax.set_title("%s — ΔF/F₀ heatmap" % name)
    ax.set_xlabel("Time (s)")
axes[0].set_ylabel("Cell number")
fig.colorbar(im, ax=axes, label="ΔF/F₀ (%)", shrink=0.8)
plt.show()

In [ ]:
N_SHOW = 12          # how many cells to stack
OFFSET = 80          # vertical spacing between stacked traces (arbitrary)

show_cells = cell_cols[:N_SHOW]

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, (name, d) in zip(axes, [("CONTROL", dff_con), ("DRUG", dff_drug)]):
    for i, c in enumerate(show_cells):
        ax.plot(d["Time"], d[c] + i * OFFSET, linewidth=0.6)   # i * OFFSET stacks them
    ax.axvline(180, color="black", linestyle="--", linewidth=1.2)
    ax.set_title("%s — individual traces (%d cells)" % (name, N_SHOW))
    ax.set_xlabel("Time (s)")
    ax.set_yticks([])                                          # the offsets are arbitrary
axes[0].set_ylabel("ΔF/F₀ (stacked, arbitrary offset)")
plt.tight_layout()
plt.show()

## Step 4 — Detect events and measure the event rate

Average ΔF/F₀ tells you how **strong** activity was. But the question about 4-AP is how **often**
cells fire, and those are not the same thing: one enormous transient and ten modest ones can average
out identically.

So we count **events**. An event is the moment a cell's ΔF/F₀ **rises above a threshold**. Two rules
keep the counting honest:

1. **Only the rising edge counts.** A transient stays above the line for dozens of frames; counting
   each frame would count one event dozens of times.
2. **Events must be at least ~1 s apart**, so a noisy signal wobbling across the line isn't counted
   three times.

Events are counted per cell in each window, then divided by the window length in minutes:

$$ \text{event rate} = \frac{\text{number of events}}{\text{window length (min)}} \qquad \text{e.g. } \frac{18 \text{ events}}{3 \text{ min}} = 6 \text{ events/min} $$

`THRESHOLD` is a judgement call — the data does not contain the answer. **You** choose it, and your
result depends on the choice. The second cell redraws the traces with a **red dot on every event**
the detector found, so you can check it by eye.

> **⚙️ Try it — raise the threshold.** Run at `THRESHOLD = 20`, then set it to `40` and re-run **both**
> cells below. Watch the rates and the dots. Set it back to **20** and re-run both.

In [ ]:
THRESHOLD = 20.0                 # ⚙️ Try it — the ΔF/F₀ (%) a cell must cross to count as an event
MIN_GAP_FRAMES = int(1.0 * FPS)  # events must be at least ~1 s apart


def event_frames(trace):
    """Frame numbers of detected events: upward crossings, kept >= MIN_GAP_FRAMES apart."""
    above  = trace > THRESHOLD
    rising = np.where((~above[:-1]) & (above[1:]))[0] + 1   # below, then above = a rising edge
    kept = []
    for r in rising:
        if not kept or r - kept[-1] >= MIN_GAP_FRAMES:
            kept.append(r)
    return np.array(kept, dtype=int)


def event_rate_per_cell(d, t_start, t_end):
    """Events per minute, for every cell, within one time window."""
    mask    = ((d["Time"] >= t_start) & (d["Time"] < t_end)).values
    minutes = (t_end - t_start) / 60.0
    return np.array([len(event_frames(d[c].values[mask])) / minutes for c in cell_cols])


rows = []
for name, d in [("CONTROL", dff_con), ("DRUG", dff_drug)]:
    base = event_rate_per_cell(d,   0, 180)      # baseline window
    app  = event_rate_per_cell(d, 180, 360)      # application window
    rows.append({"Group": name,
                 "Baseline (events/min)":    round(base.mean(), 2),
                 "Application (events/min)": round(app.mean(), 2)})

print("THRESHOLD = %g %% ΔF/F₀\n" % THRESHOLD)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
def event_peaks(trace):
    """Same events as event_frames, nudged to each transient's peak so the dots sit on top."""
    win = int(1.5 * FPS)
    return np.array([r + int(np.argmax(trace[r:r + win])) for r in event_frames(trace)], dtype=int)


fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, (name, d) in zip(axes, [("CONTROL", dff_con), ("DRUG", dff_drug)]):
    t = d["Time"].values
    for i, c in enumerate(show_cells):            # same cells as Step 3
        trace = d[c].values
        ax.plot(t, trace + i * OFFSET, linewidth=0.6, color="0.6")
        ev = event_peaks(trace)
        ax.plot(t[ev], trace[ev] + i * OFFSET, "o", color="crimson", markersize=3, zorder=3)
    ax.axvline(180, color="black", linestyle="--", linewidth=1.2)
    ax.set_title("%s — detected events (THRESHOLD = %g%%)" % (name, THRESHOLD))
    ax.set_xlabel("Time (s)")
    ax.set_yticks([])
axes[0].set_ylabel("ΔF/F₀ (stacked) — dots mark detected events")
plt.tight_layout()
plt.show()

### Your results

Every number the Discussion asks for, in one table, at the default settings.

In [ ]:
if F0_WINDOW_SEC != 180 or THRESHOLD != 20:
    print("*** Set F0_WINDOW_SEC back to 180 (Step 2) and THRESHOLD back to 20 (Step 4), ***")
    print("*** then re-run from Step 2 so this table shows the default numbers.        ***\n")

result_rows = []
for name, d in [("CONTROL", dff_con), ("DRUG", dff_drug)]:
    base_rate = event_rate_per_cell(d,   0, 180)
    app_rate  = event_rate_per_cell(d, 180, 360)
    result_rows.append({
        "Group": name,
        "Application_mean_dFF_pct":   round(d.loc[d["Time"] >= 180, cell_cols].values.mean(), 2),
        "Baseline_events_per_min":    round(base_rate.mean(), 2),
        "Application_events_per_min": round(app_rate.mean(), 2),
    })

my_result = pd.DataFrame(result_rows).set_index("Group")
my_result.astype(object).T      # .T flips the table on its side so it is easier to read

## Putting it all together

| Step | Idea | What it gave you |
|---|---|---|
| 1 | Raw fluorescence | brightness differs between cells for reasons unrelated to activity |
| 2 | $\Delta F/F_0 = (F - F_0)/F_0$ | every cell on its own scale, so cells can be compared |
| 3 | Heatmap + traces | **how many** cells responded and **when**; the **shape** of each event |
| 4 | Event rate | how **often** cells fired — and how much that depended on `THRESHOLD` |

**But this is one slice per condition.** The 100 cells in one recording are not 100 independent
experiments — they share a slice, a wash, and a day. Before anyone could claim that 4-AP "works" you
would need **several slices per group**, and a test that says whether the difference between groups is
bigger than chance would produce anyway. That test is where **W11 L2** picks up.

---

### 💬 Discussion

Keep each answer **short** (a number and one sentence is often enough). Use the default ⚙️ values
unless a question says otherwise.

1. **(Step 1)** How many rows does **one** recording have, and how many does the combined table have?
   What was the resting brightness of cell **C4** and of cell **C5**? Why can't you tell from those
   numbers which cell is more active?

2. **(Step 2)** Report the **application** ΔF/F₀ for CONTROL and for DRUG. When you set
   `F0_WINDOW_SEC = 360`, what did the DRUG **baseline** become, and why did it stop being 0%?

3. **(Step 3)** After the dashed line, what changes in the DRUG heatmap that does **not** change in
   CONTROL? When you set `COLOUR_MAX` to 20 and then 150, did any number in the data change?

4. **(Step 4)** Report the four event rates at `THRESHOLD = 20` (CONTROL baseline and application,
   DRUG baseline and application). At `THRESHOLD = 40`, what was the DRUG application rate — and did
   the conclusion that 4-AP raises the event rate survive? Why is one slice per group still not
   enough to conclude that 4-AP works?

*(Before you submit: set `CELL_TO_INSPECT = "C1"`, `F0_WINDOW_SEC = 180`, `COLOUR_MAX = 60`,
and `THRESHOLD = 20`, then run all.)*

**Your answer:** *(double-click here to type)*

>
>
>
>

---
### Nice work — you're done!

Click **Runtime → Run all**, then **File → Print → Save as PDF**, and upload the PDF to Canvas.

This cell is empty on purpose. Do not delete.